#### Cell 1 — Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import random

#### Cell 2 — Configuration

In [0]:
random.seed(42)

TABLE_NAME = "banking_risk_demo.bronze_merchants_raw"
NUM_MERCHANTS = 300

merchant_categories = [
    "grocery",
    "travel",
    "electronics",
    "fuel",
    "hospitality",
    "utilities",
    "ecommerce",
    "healthcare",
    "entertainment"
]

merchant_countries = ["AU", "NZ", "SG", "GB", "US", "JP", "DE"]
merchant_risk_ratings = ["low", "medium", "high"]

merchant_name_prefixes = [
    "Global", "Prime", "Metro", "Blue", "Summit", "Urban", "National", "Pacific",
    "Vertex", "Orbit", "Swift", "Harbour", "Elite", "Core", "Next", "Central"
]

merchant_name_suffixes = [
    "Retail", "Stores", "Services", "Market", "Travel", "Fuel", "Electronics",
    "Hub", "Trading", "Outlet", "Solutions", "Foods", "Group", "Direct"
]

#### Cell 3 — Create helper functions

In [0]:
def generate_merchant_id(i: int) -> str:
    return f"MERCH{i:04d}"

def generate_merchant_name() -> str:
    return f"{random.choice(merchant_name_prefixes)} {random.choice(merchant_name_suffixes)}"

def weighted_choice(options, weights):
    return random.choices(options, weights=weights, k=1)[0]

#### Cell 4 — Generate mostly valid merchant records

In [0]:
merchants = []

for i in range(1, NUM_MERCHANTS + 1):
    merchant_id = generate_merchant_id(i)
    merchant_name = generate_merchant_name()
    
    merchant_category = weighted_choice(
        merchant_categories,
        [0.18, 0.10, 0.10, 0.12, 0.12, 0.10, 0.18, 0.05, 0.05]
    )
    
    merchant_country = weighted_choice(
        merchant_countries,
        [0.65, 0.08, 0.06, 0.06, 0.06, 0.05, 0.04]
    )
    
    merchant_risk_rating = weighted_choice(
        ["low", "medium", "high"],
        [0.60, 0.30, 0.10]
    )
    
    sanctions_flag = weighted_choice(
        ["Y", "N"],
        [0.01, 0.99]
    )

    merchants.append({
        "merchant_id": merchant_id,
        "merchant_name": merchant_name,
        "merchant_category": merchant_category,
        "merchant_country": merchant_country,
        "merchant_risk_rating": merchant_risk_rating,
        "sanctions_flag": sanctions_flag
    })

len(merchants)

300

#### Cell 5 — Inject intentional bad data

In [0]:
# 1. Duplicate some merchant IDs
for idx in random.sample(range(NUM_MERCHANTS), 4):
    duplicate_source = random.randint(0, NUM_MERCHANTS - 1)
    merchants[idx]["merchant_id"] = merchants[duplicate_source]["merchant_id"]

# 2. Null merchant_name for some records
for idx in random.sample(range(NUM_MERCHANTS), 6):
    merchants[idx]["merchant_name"] = None

# 3. Null merchant_category for some records
for idx in random.sample(range(NUM_MERCHANTS), 10):
    merchants[idx]["merchant_category"] = None

# 4. Invalid merchant_country values
for idx in random.sample(range(NUM_MERCHANTS), 5):
    merchants[idx]["merchant_country"] = random.choice(["XX", "", "UNKNOWN"])

# 5. Invalid merchant_risk_rating values
for idx in random.sample(range(NUM_MERCHANTS), 5):
    merchants[idx]["merchant_risk_rating"] = random.choice(["severe", "unknown"])

# 6. Invalid sanctions_flag values
for idx in random.sample(range(NUM_MERCHANTS), 3):
    merchants[idx]["sanctions_flag"] = random.choice(["maybe", "", "X"])

#### Cell 6 — Convert to Spark DataFrame

In [0]:
merchant_schema = T.StructType([
    T.StructField("merchant_id", T.StringType(), True),
    T.StructField("merchant_name", T.StringType(), True),
    T.StructField("merchant_category", T.StringType(), True),
    T.StructField("merchant_country", T.StringType(), True),
    T.StructField("merchant_risk_rating", T.StringType(), True),
    T.StructField("sanctions_flag", T.StringType(), True),
])

merchants_df = spark.createDataFrame(merchants, schema=merchant_schema)

display(merchants_df.limit(20))

merchant_id,merchant_name,merchant_category,merchant_country,merchant_risk_rating,sanctions_flag
MERCH0001,Blue Retail,ecommerce,AU,low,N
MERCH0002,Metro Outlet,fuel,AU,low,N
MERCH0003,Global Trading,travel,AU,low,N
MERCH0004,Vertex Group,ecommerce,SG,low,N
MERCH0005,Vertex Services,travel,SG,low,N
MERCH0006,Harbour Direct,electronics,AU,low,N
MERCH0007,Blue Electronics,grocery,AU,medium,N
MERCH0008,Harbour Outlet,travel,AU,medium,N
MERCH0009,Metro Direct,travel,AU,unknown,N
MERCH0010,Harbour Services,electronics,AU,low,N


#### Cell 7 — Save the Bronze table

In [0]:
(
    merchants_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(TABLE_NAME)
)

#### Cell 8 — Validate the saved table

In [0]:
saved_df = spark.table(TABLE_NAME)

print("Row count:", saved_df.count())

display(saved_df.limit(20))

Row count: 300


merchant_id,merchant_name,merchant_category,merchant_country,merchant_risk_rating,sanctions_flag
MERCH0001,Blue Retail,ecommerce,AU,low,N
MERCH0002,Metro Outlet,fuel,AU,low,N
MERCH0003,Global Trading,travel,AU,low,N
MERCH0004,Vertex Group,ecommerce,SG,low,N
MERCH0005,Vertex Services,travel,SG,low,N
MERCH0006,Harbour Direct,electronics,AU,low,N
MERCH0007,Blue Electronics,grocery,AU,medium,N
MERCH0008,Harbour Outlet,travel,AU,medium,N
MERCH0009,Metro Direct,travel,AU,unknown,N
MERCH0010,Harbour Services,electronics,AU,low,N


#### Cell 9 — Run quick diagnostic checks

In [0]:
from pyspark.sql.functions import col, count, when

print("Duplicate merchant_id values:")
display(
    saved_df.groupBy("merchant_id")
    .count()
    .filter(col("count") > 1)
)

print("Null merchant_name count:")
display(
    saved_df.select(count(when(col("merchant_name").isNull(), 1)).alias("null_merchant_name_count"))
)

print("Null merchant_category count:")
display(
    saved_df.select(count(when(col("merchant_category").isNull(), 1)).alias("null_merchant_category_count"))
)

print("Invalid merchant_country values:")
display(
    saved_df.filter(~col("merchant_country").isin("AU", "NZ", "SG", "GB", "US", "JP", "DE"))
)

print("Invalid merchant_risk_rating values:")
display(
    saved_df.filter(~col("merchant_risk_rating").isin("low", "medium", "high"))
)

print("Invalid sanctions_flag values:")
display(
    saved_df.filter(~col("sanctions_flag").isin("Y", "N"))
)

Duplicate merchant_id values:


merchant_id,count
MERCH0286,2
MERCH0192,2
MERCH0174,2
MERCH0188,2


Null merchant_name count:


null_merchant_name_count
6


Null merchant_category count:


...

merchant_id,merchant_name,merchant_category,merchant_country,merchant_risk_rating,sanctions_flag
MERCH0084,Next Electronics,grocery,UNKNOWN,low,N
MERCH0154,Prime Travel,electronics,UNKNOWN,severe,N
MERCH0164,Summit Solutions,grocery,,low,N
MERCH0168,Orbit Foods,ecommerce,UNKNOWN,low,N
MERCH0188,Orbit Fuel,electronics,UNKNOWN,low,N


Invalid merchant_risk_rating values:


merchant_id,merchant_name,merchant_category,merchant_country,merchant_risk_rating,sanctions_flag
MERCH0009,Metro Direct,travel,AU,unknown,N
MERCH0154,Prime Travel,electronics,UNKNOWN,severe,N
MERCH0244,Vertex Hub,healthcare,AU,unknown,N
MERCH0251,Next Stores,grocery,US,unknown,N
MERCH0289,Orbit Solutions,entertainment,AU,unknown,N


Invalid sanctions_flag values:


merchant_id,merchant_name,merchant_category,merchant_country,merchant_risk_rating,sanctions_flag
MERCH0014,Swift Market,utilities,AU,high,X
MERCH0136,Orbit Hub,ecommerce,AU,low,X
MERCH0243,Blue Foods,null,US,low,X
